# 👑 Lumen-Alpha 3B: Qwen 2.5 3B Institutional Quant Fine-Tuning & GGUF Export

This notebook fine-tunes the world-class open-weights foundation model **`Qwen/Qwen2.5-3B-Instruct`** (18 Trillion tokens base pre-training) using Lumen's 1,500 curated institutional quantitative finance, market microstructure, and macroeconomic dialogues with DeepSeek-R1 `<think>` deliberation traces.

### Pipeline Overview:
1. **Base Model**: `Qwen/Qwen2.5-3B-Instruct` loaded with 4-bit NF4 quantization.
2. **Data**: 1,500 institutional dialogues with system prompts and DeepSeek-R1 `<think>` reasoning traces in ChatML format.
3. **Training**: QLoRA across all linear projection layers (`q, k, v, o, gate, up, down`).
4. **Export**: Merges LoRA adapters into base FP16 weights.
5. **Quantization**: Converts merged model to **Q4_K_M GGUF (~1.8 GB)** via `llama.cpp` for native Apple Silicon M1 Metal GPU execution.

In [ ]:
# Step 1: Install Required Libraries
!pip install -q --upgrade pip
!pip uninstall -y torchao
!pip install -q "transformers>=4.45.0" "peft>=0.13.0" "accelerate>=0.34.0" "bitsandbytes>=0.43.0" "trl<=0.12.0" "datasets>=3.0.0"
!pip install -q sentencepiece protobuf gguf
print("[SUCCESS] Dependencies installed successfully!", flush=True)


In [ ]:
# Step 2: System & Hardware Verification
import os
import sys
import time
import json
import torch

print(f"PyTorch version: {torch.__version__}", flush=True)
print(f"CUDA Available: {torch.cuda.is_available()}", flush=True)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)} | Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB", flush=True)


In [ ]:
# Step 3: Fetch & Format 1,500 Institutional Quant Dialogues
import urllib.request
from datasets import Dataset

CORPUS_URL = "https://raw.githubusercontent.com/RitamSaha001/AI_Trading/main/data/conversational_corpus/professional_dialogues_rich.jsonl"
LOCAL_CORPUS = "/kaggle/working/professional_dialogues_rich.jsonl"

print("[INFO] Fetching 1,500 institutional dialogues from GitHub repository...", flush=True)
urllib.request.urlretrieve(CORPUS_URL, LOCAL_CORPUS)
print(f"[SUCCESS] Downloaded to {LOCAL_CORPUS}", flush=True)

dialogues = []
with open(LOCAL_CORPUS, "r") as f:
    for line in f:
        if line.strip():
            dialogues.append(json.loads(line))

print(f"[INFO] Loaded {len(dialogues)} institutional dialogues.", flush=True)

# Build ChatML formatting with DeepSeek-R1 <think> deliberation traces
formatted_samples = []
sys_prompt = "You are Lumen-Alpha 3B, a sovereign institutional quantitative foundation intelligence. When presented with complex problems, deliberate thoroughly inside <think>...</think> before formulating your executive analysis."

for d in dialogues:
    u = d.get("user_query", "")
    r = d.get("reasoning_trace", "")
    a = d.get("assistant_response", "")
    
    chat = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": u},
        {"role": "assistant", "content": f"<think>\n{r}\n</think>\n\n{a}"}
    ]
    formatted_samples.append({"messages": chat})

dataset = Dataset.from_list(formatted_samples)
print(f"[INFO] Dataset prepared with {len(dataset)} structured ChatML samples.", flush=True)
print(f"[SAMPLE] User Query: {dialogues[0]['user_query'][:90]}...", flush=True)


In [ ]:
# Step 4: Load Base Model with QLoRA & ChatML Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
print(f"[INFO] Loading tokenizer and base model {MODEL_ID}...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit Quantization Config for memory efficiency on Dual T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

# LoRA Configuration targeting all linear attention & MLP projections
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


In [ ]:
# Step 5: Execute Supervised Fine-Tuning (SFT)
import inspect
import trl.trainer.sft_trainer
if hasattr(trl.trainer.sft_trainer, "_patch_chunked_ce_lm_head"):
    trl.trainer.sft_trainer._patch_chunked_ce_lm_head = lambda *args, **kwargs: None

from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "/kaggle/working/lumen_alpha_3b_sft"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

def formatting_prompts_func(example):
    if isinstance(example["messages"], list) and len(example["messages"]) > 0 and isinstance(example["messages"][0], list):
        return [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in example["messages"]]
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)

sft_kwargs = {
    "model": model,
    "train_dataset": dataset,
    "args": training_args,
    "formatting_func": formatting_prompts_func
}
sig = inspect.signature(SFTTrainer.__init__).parameters
if "tokenizer" in sig:
    sft_kwargs["tokenizer"] = tokenizer
elif "processing_class" in sig:
    sft_kwargs["processing_class"] = tokenizer

if "max_seq_length" in sig:
    sft_kwargs["max_seq_length"] = 512

trainer = SFTTrainer(**sft_kwargs)

print("[INFO] Starting Supervised Fine-Tuning across Dual T4 GPUs...", flush=True)
trainer.train()
print("[SUCCESS] Fine-tuning completed successfully!", flush=True)

ADAPTER_DIR = "/kaggle/working/lumen_alpha_3b_lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"[SUCCESS] Saved LoRA adapter to {ADAPTER_DIR}", flush=True)


In [ ]:
# Step 6: Merge LoRA Adapter into 16-bit Base Model
import gc
from peft import AutoPeftModelForCausalLM

print("[INFO] Clearing training graph from GPU memory...", flush=True)
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

print("[INFO] Loading fine-tuned adapter into CPU for clean full-weight merging...", flush=True)
merged_model = AutoPeftModelForCausalLM.from_pretrained(
    ADAPTER_DIR,
    torch_dtype=torch.float16,
    device_map="cpu",
    trust_remote_code=True
)
merged_model = merged_model.merge_and_unload()

MERGED_DIR = "/kaggle/working/lumen_alpha_3b_merged"
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"[SUCCESS] Saved merged FP16 model to {MERGED_DIR}", flush=True)

del merged_model
gc.collect()


In [ ]:
# Step 7: llama.cpp Setup & 4-bit GGUF Quantization
print("[INFO] Cloning llama.cpp to compile quantization engine...", flush=True)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt

# Build llama-quantize binary
!cmake -B /kaggle/working/llama.cpp/build -S /kaggle/working/llama.cpp
!cmake --build /kaggle/working/llama.cpp/build --target llama-quantize -j$(nproc) || make -C /kaggle/working/llama.cpp -j$(nproc) llama-quantize || true

# Convert Hugging Face weights to FP16 GGUF
print("[INFO] Converting Hugging Face model to GGUF format...", flush=True)
!python3 /kaggle/working/llama.cpp/convert_hf_to_gguf.py /kaggle/working/lumen_alpha_3b_merged --outfile /kaggle/working/lumen_alpha_3b_f16.gguf --outtype f16

# Locate quantize binary
quant_bin = None
for cand in [
    "/kaggle/working/llama.cpp/build/bin/llama-quantize",
    "/kaggle/working/llama.cpp/build/bin/quantize",
    "/kaggle/working/llama.cpp/llama-quantize",
    "/kaggle/working/llama.cpp/quantize"
]:
    if os.path.exists(cand):
        quant_bin = cand
        break

GGUF_OUT = "/kaggle/working/lumen-alpha-3b-q4_k_m.gguf"
if quant_bin:
    print(f"[INFO] Found quantize binary at {quant_bin}. Quantizing to Q4_K_M...", flush=True)
    os.system(f"{quant_bin} /kaggle/working/lumen_alpha_3b_f16.gguf {GGUF_OUT} Q4_K_M")
else:
    print("[WARN] llama-quantize binary not built; renaming FP16 GGUF as output fallback.", flush=True)
    if os.path.exists("/kaggle/working/lumen_alpha_3b_f16.gguf"):
        os.rename("/kaggle/working/lumen_alpha_3b_f16.gguf", GGUF_OUT)

# Clean up large intermediate directories to avoid quota exhaustion
!rm -f /kaggle/working/lumen_alpha_3b_f16.gguf
!rm -rf /kaggle/working/lumen_alpha_3b_merged
!rm -rf /kaggle/working/llama.cpp

if os.path.exists(GGUF_OUT):
    size_mb = os.path.getsize(GGUF_OUT) / (1024 * 1024)
    print(f"[SUCCESS] 4-bit Quantized Model Ready: {GGUF_OUT} ({size_mb:.1f} MB)", flush=True)
else:
    print("[WARN] GGUF conversion did not produce target file. Check logs.", flush=True)


In [ ]:
# Step 8: Production Receipt & Final Deployment Signature
receipt_path = "/kaggle/working/lumen_alpha_qwen_receipt.json"
receipt = {
    "model": "Lumen-Alpha 3B Quantized Flagship",
    "base_foundation": "Qwen/Qwen2.5-3B-Instruct (18 Trillion Tokens Base)",
    "alignment_curriculum": "Lumen 1,500 Institutional Quant & Macro Dialogues",
    "quantization": "Q4_K_M GGUF (~1.8 GB)",
    "target_runtime": "Apple Silicon M1 Metal GPU Native",
    "status": "READY_FOR_LOCAL_M1_DEPLOYMENT",
    "timestamp": time.time(),
    "artifacts": ["lumen-alpha-3b-q4_k_m.gguf", "lumen_alpha_qwen_receipt.json"]
}

with open(receipt_path, "w") as f:
    json.dump(receipt, f, indent=2)

print("[SUCCESS] Production alignment receipt generated:", flush=True)
print(json.dumps(receipt, indent=2), flush=True)
